# 04 Expand Abbreviations

Reads `processed/03_spelling.csv` and expands standard abbreviations ("dr" becomes "doctor", "rd" becomes "road"). Internet slang ("lol", "idk") is left alone.

Only abbreviations that can be expanded safely are included:

- Titles: dr, mr, mrs, jr, sr. `ms` is left alone because it is also Mississippi ("jackson, ms").
- Street words: rd, ave, blvd.
- `st` is "saint" when a name follows ("st. patrick") and "street" otherwise.
- Units: oz and fl oz (only after a number or "fl", so "wizard of oz" stays), lb/lbs, ft after a number ("5280 ft" is feet; "ft. knox" is fort), mph.
- A few common short forms: tv, bbq, dept, approx, avg.

Left for the stop word step because they are not content words: etc, eg, ie, vs.

Output: `processed/04_abbreviations.csv` and `processed/04_abbreviation_changes.csv` (how often each rule fired).

In [1]:
import pandas as pd
import re
from collections import Counter

In [2]:
data = pd.read_csv("processed/03_spelling.csv", dtype=str, keep_default_na=False)
print(f"{len(data):,} rows")

185,824 rows


In [3]:
# (name, pattern, replacement); a trailing period belongs to the abbreviation and is removed with it
NUM = r"(?<=\d)\s*|(?<=\d\s)"   # after a number, with or without a space
RULES = [
    ("dr",     r"\bdr\b\.?",                       "doctor"),
    ("mr",     r"\bmr\b\.?",                       "mister"),
    ("mrs",    r"\bmrs\b\.?",                      "missus"),
    ("jr",     r"\bjr\b\.?",                       "junior"),
    ("sr",     r"\bsr\b\.?",                       "senior"),
    ("rd",     r"\brd\b\.?",                       "road"),
    ("ave",    r"\bave\b\.?",                      "avenue"),
    ("blvd",   r"\bblvd\b\.?",                     "boulevard"),
    ("st_saint",  r"\bst\b\.?(?=\s+[a-z])",        "saint"),
    ("st_street", r"\bst\b\.?",                     "street"),
    ("fl_oz",  r"\bfl\.?\s*oz\b\.?",              "fluid ounce"),
    ("oz",     r"(?<=\d)\s*oz\b\.?|(?<=\d )oz\b\.?", " ounce"),
    ("lbs",    r"\blbs\b\.?",                      "pounds"),
    ("lb",     r"\blb\b\.?",                       "pound"),
    ("ft_fort", r"\bft\b\.?(?=\s+[a-z])",         "fort"),
    ("ft_feet", r"(?<=\d)\s*ft\b\.?|(?<=\d )ft\b\.?", " feet"),
    ("mph",    r"\bmph\b",                          "miles per hour"),
    ("tv",     r"\btv\b\.?",                       "television"),
    ("bbq",    r"\bbbq\b",                          "barbecue"),
    ("dept",   r"\bdept\b\.?",                     "department"),
    ("approx", r"\bapprox\b\.?",                   "approximately"),
    ("avg",    r"\bavg\b\.?",                      "average"),
]
COMPILED = [(name, re.compile(pat), rep) for name, pat, rep in RULES]
fired = Counter()

def expand(text):
    for name, pat, rep in COMPILED:
        text, n = pat.subn(rep, text)
        if n:
            fired[name] += n
    return text

In [4]:
data["answer_abbreviations"] = data["answer_spelled"].map(expand)
n_changed = (data["answer_abbreviations"] != data["answer_spelled"]).sum()
print(f"{n_changed:,} answers changed")

log = pd.DataFrame(fired.items(), columns=["rule", "count"]).sort_values("count", ascending=False)
log.to_csv("processed/04_abbreviation_changes.csv", index=False)
log

665 answers changed


,rule,count
0,tv,447
10,st_saint,66
1,bbq,49
6,mr,28
5,dr,27
7,mrs,20
14,lbs,12
2,jr,9
17,mph,7
4,oz,6


Examples of each rule, to check:

In [5]:
pd.set_option("display.max_colwidth", 110)
changed = data[data["answer_abbreviations"] != data["answer_spelled"]]
for name in ["dr", "mr", "st_saint", "st_street", "oz", "ft_fort", "ft_feet", "tv"]:
    pat = dict((n, p) for n, p, _ in COMPILED)[name]
    rows = changed[changed["answer_spelled"].map(lambda t: bool(pat.search(t)))].head(2)
    for _, r in rows.iterrows():
        m = pat.search(r["answer_spelled"])
        a, b = max(0, m.start() - 20), m.end() + 20
        print(f"{name:10s} {r['answer_spelled'][a:b]!r}  ->  {expand(r['answer_spelled'][a:b])!r}")

dr         'them, it, dr. sets, who-ville, ho'  ->  'them, it, doctor sets, who-ville, ho'
dr         ', miss, ms., madam, dr., profession'  ->  ', miss, ms., madam, doctor, profession'
mr         'name, mr., sir, mrs., miss, m'  ->  'name, mister, sir, missus, miss, m'
mr         'mr. mrs. miss. ms. dr.'  ->  'mister missus miss. ms. doctor'
st_saint   'ockings, christmas, st. nicholas, mrs. clau'  ->  'ockings, christmas, saint nicholas, missus clau'
st_saint   'led, christmas eve, st nick'  ->  'led, christmas eve, saint nick'
st_street  'ockings, christmas, st. nicholas, mrs. clau'  ->  'ockings, christmas, saint nicholas, missus clau'
st_street  'led, christmas eve, st nick'  ->  'led, christmas eve, saint nick'
oz         '8oz, fl oz, glass, plas'  ->  '8 ounce, fluid ounce, glass, plas'
oz         't of measurement. 32oz. four cups make up a'  ->  't of measurement. 32 ounce four cups make up a'
ft_fort    'ups and running. at ft. knox. basic cable i'  ->  'ups and running. at f

In [6]:
out = data[["row_id", "subject_id", "cue", "answer_spelled", "answer_abbreviations"]]
out.to_csv("processed/04_abbreviations.csv", index=False)
print(f"Saved {len(out):,} rows to processed/04_abbreviations.csv")

Saved 185,824 rows to processed/04_abbreviations.csv
